In [1]:
"function (and parameter space) definitions for hyperband"
"regression with Keras (multilayer perceptron)"

import numpy as np
import matplotlib.pyplot as plt
import pickle

from hyperopt import hp
from hyperopt.pyll.stochastic import sample

from math import log, sqrt
from time import time
from pprint import pprint

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score as AUC, log_loss, accuracy_score as accuracy
from sklearn.metrics import mean_squared_error as MSE, mean_absolute_error as MAE, r2_score as R2, explained_variance_score as EVS
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler

from keras.models import Sequential
from keras.layers.core import Dense, Dropout
from keras.layers.normalization import BatchNormalization as BatchNorm
from keras.callbacks import EarlyStopping
from keras.layers.advanced_activations import *

# TODO: advanced activations - 'leakyrelu', 'prelu', 'elu', 'thresholdedrelu', 'srelu' 

%load_ext autoreload
%autoreload 2

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 8)

Using TensorFlow backend.


In [2]:
with open('dict_of_arrays.pickle', 'rb') as f:
    alldata = pickle.load(f)
    
print(alldata['xs'].shape)
print(alldata['ys'].shape)

# Check the units, convert to MJ...
print(alldata['ys'][:5])
print(alldata['ys'][:5] * (60/10**6))

alldata['ys'] = alldata['ys'] * (60/10**6)

(80149, 5)
(80149,)
[ 32980.6  84242.8  99181.1  74662.2  43414. ]
[ 1.978836  5.054568  5.950866  4.479732  2.60484 ]


In [3]:
x_train, x_test, y_train, y_test = train_test_split(alldata['xs'], alldata['ys'], test_size=0.15)
data = {'x_train':x_train, 'x_test':x_test, 'y_train':y_train, 'y_test':y_test}

In [4]:
print(x_train[:10]) # enlem - parlak saatler - ortalama sıcaklık - gün uzunluğu - h0

[[  3.67372000e+01   1.02000000e+01   4.91250000e+00   1.17179366e+01
    2.86565009e+07]
 [  3.91436000e+01   1.01000000e+01   1.59875000e+01   1.38018928e+01
    3.83736911e+07]
 [  3.75480000e+01   7.80000000e+00   2.09166667e+00   1.06049974e+01
    2.17139603e+07]
 [  3.97769000e+01   8.20000000e+00   9.34583333e+00   1.37347888e+01
    3.78240997e+07]
 [  3.94983000e+01   5.40000000e+00   1.06583333e+01   1.13415611e+01
    2.54197453e+07]
 [  3.68406000e+01   5.00000000e-01   1.07083333e+01   1.11230376e+01
    2.50274563e+07]
 [  3.80240000e+01   4.60000000e+00   1.43521739e+01   1.21069948e+01
    3.00984523e+07]
 [  3.87380000e+01   9.70000000e+00   1.89958333e+01   1.46494053e+01
    4.15753898e+07]
 [  3.90788000e+01   6.90000000e+00   1.59166667e+01   1.16893673e+01
    2.73146571e+07]
 [  3.86237000e+01   0.00000000e+00   8.05833333e+00   1.02821190e+01
    1.95884823e+07]]


In [5]:
# handle floats which should be integers
# works with flat params
def handle_integers( params ):
    new_params = {}
    for k, v in params.items():
        if type( v ) == float and int( v ) == v:
            new_params[k] = int( v )
        else:
            new_params[k] = v
    
    return new_params

In [18]:
max_layers = 4
max_layer_size = 20

space = {
    'scaler': hp.choice( 's', 
        ( None, 'StandardScaler', 'RobustScaler', 'MinMaxScaler', 'MaxAbsScaler' )),
    'n_layers': hp.quniform( 'ls', 1, max_layers, 1 ),
    'init': hp.choice( 'i', ( 'uniform', 'normal', 'glorot_uniform', 
        'glorot_normal', 'he_uniform', 'he_normal' )),
    'batch_size': hp.choice( 'bs', ( 16, 32, 64, 128 )),
    'shuffle': hp.choice( 'sh', ( False, True )),
    'loss': hp.choice( 'l', ( 'mean_absolute_error', 'mean_squared_error' )),
    'optimizer': hp.choice( 'o', ( 'rmsprop', 'adagrad', 'adadelta', 'adam', 'adamax' ))        
}

# for each hidden layer, we choose size, activation and extras individually
for i in range( 1, max_layers + 1 ):
    space[ 'layer_{}_size'.format( i )] = hp.quniform( 'ls{}'.format( i ), 
        2, max_layer_size, 1 )
    space[ 'layer_{}_activation'.format( i )] = hp.choice( 'a{}'.format( i ), 
        ( 'relu', 'sigmoid', 'tanh' ))
    space[ 'layer_{}_extras'.format( i )] = hp.choice( 'e{}'.format( i ), ( 
        { 'name': 'dropout', 'rate': hp.uniform( 'd{}'.format( i ), 0.1, 0.5 )}, 
        { 'name': 'batchnorm' },
        { 'name': None } ))    
    
def get_params():
    params = sample( space )
    return handle_integers( params )

# print hidden layers config in readable way
def print_layers( params ):
    for i in range( 1, params['n_layers'] + 1 ):
        print("layer {} | size: {:>3} | activation: {:<7} | extras: {}".format( i,
            params['layer_{}_size'.format( i )], 
            params['layer_{}_activation'.format( i )],
            params['layer_{}_extras'.format( i )]['name'] ))
        if params['layer_{}_extras'.format( i )]['name'] == 'dropout':
            print("- rate: {:.1%}".format( params['layer_{}_extras'.format( i )]['rate'] ))

def print_params( params ):
    pprint({ k: v for k, v in params.items() if not k.startswith( 'layer_' )})
    print_layers( params )

def try_params( n_iterations, params ):
    
    print("iterations:", n_iterations)
    print_params( params )
    
    y_train = data['y_train']
    y_test = data['y_test']
        
    if params['scaler']:
        scaler = eval( "{}()".format( params['scaler'] ))
        x_train_ = scaler.fit_transform( data['x_train'].astype( float ))
        x_test_ = scaler.transform( data['x_test'].astype( float ))
    else:
        x_train_ = data['x_train']
        x_test_ = data['x_test']
        
    input_dim = x_train_.shape[1]

    model = Sequential()
    model.add( Dense( params['layer_1_size'], kernel_initializer = params['init'], 
        activation = params['layer_1_activation'], input_dim = input_dim ))
    
    for i in range( int( params['n_layers'] ) - 1 ):
        
        extras = 'layer_{}_extras'.format( i + 1 )
        
        if params[extras]['name'] == 'dropout':
            model.add( Dropout( params[extras]['rate'] ))
        elif params[extras]['name'] == 'batchnorm':
            model.add( BatchNorm())
            
        model.add( Dense( params['layer_{}_size'.format( i + 2 )], kernel_initializer = params['init'], 
            activation = params['layer_{}_activation'.format( i + 2 )]))
           
    model.add( Dense( 1, kernel_initializer = params['init'], activation = 'linear' ))

    model.compile( optimizer = params['optimizer'], loss = params['loss'] )
    
    validation_data = ( x_test_, y_test )

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 5, verbose = 0 )
    
    history = model.fit( x_train_, y_train,
        epochs = int( round( n_iterations )),
        batch_size = params['batch_size'], 
        shuffle = params['shuffle'], 
        validation_data = validation_data, 
        callbacks = [ early_stopping ])    
    
    p = model.predict( x_train_, batch_size = params['batch_size'] )

    mse = MSE( y_train, p )
    rmse = sqrt( mse )
    mae = MAE( y_train, p )
    r2 = R2( y_train, p )
    evs = EVS( y_train, p )

    print("\n# training | RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}".format( rmse, mae, r2, evs ))

    p = model.predict( x_test_, batch_size = params['batch_size'] )
    
    mse = MSE( y_test, p )
    rmse = sqrt( mse )
    mae = MAE( y_test, p )
    r2 = R2( y_test, p )
    evs = EVS( y_test, p )

    print("\n# test | RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}".format( rmse, mae, r2, evs ))
    
    return { 'loss': rmse, 'rmse': rmse, 'mae': mae, 'r2':r2, 'evs':evs, 'early_stop': model.stop_training }


In [19]:
from hyperband import Hyperband

output_file = 'results.pkl'
print("Will save results to", output_file)

hb = Hyperband( get_params, try_params )
results = hb.run( skip_last = 1 )

print("{} total, best:\n".format( len( results )))

for r in sorted( results, key = lambda x: x['loss'] )[:5]:
    print("loss: {:.2%} | {} seconds | {:.1f} iterations | run {} ".format( 
        r['loss'], r['seconds'], r['iterations'], r['counter'] ))
    pprint( r['params'] )

print("saving...")

with open( output_file, 'wb' ) as f:
    pickle.dump( results, f )

Will save results to results.pkl

*** 81 configurations x 1.0 iterations each

1 | Fri Jan 26 13:56:46 2018 | lowest loss so far: inf (run -1)

iterations: 1.0
{'batch_size': 128,
 'init': 'glorot_normal',
 'loss': 'mean_squared_error',
 'n_layers': 2,
 'optimizer': 'adagrad',
 'scaler': 'RobustScaler',
 'shuffle': False}
layer 1 | size:  24 | activation: sigmoid | extras: None
layer 2 | size:  36 | activation: sigmoid | extras: dropout
- rate: 44.2%
Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 1s - loss: 170.5042 - val_loss: 106.2103

# training | RMSE: 10.3111, MAE: 8.3629, R2: -0.4429, EVS: 0.0424

# test | RMSE: 10.3058, MAE: 8.3583, R2: -0.4353, EVS: 0.0423

3 seconds.

2 | Fri Jan 26 13:56:49 2018 | lowest loss so far: 10.3058 (run 1)

iterations: 1.0
{'batch_size': 32,
 'init': 'glorot_normal',
 'loss': 'mean_squared_error',
 'n_layers': 3,
 'optimizer': 'adadelta',
 'scaler': 'MinMaxScaler',
 'shuffle': True}
layer 1

68126/68126 [==============================] - 5s - loss: 7.3929 - val_loss: 3.3652

# training | RMSE: 4.4689, MAE: 3.3995, R2: 0.7290, EVS: 0.7291

# test | RMSE: 4.3958, MAE: 3.3652, R2: 0.7389, EVS: 0.7390

9 seconds.

25 | Fri Jan 26 14:01:16 2018 | lowest loss so far: 4.1348 (run 20)

iterations: 1.0
{'batch_size': 32,
 'init': 'he_normal',
 'loss': 'mean_absolute_error',
 'n_layers': 3,
 'optimizer': 'adadelta',
 'scaler': None,
 'shuffle': True}
layer 1 | size:  25 | activation: relu    | extras: None
layer 2 | size:  31 | activation: tanh    | extras: None
layer 3 | size:  31 | activation: sigmoid | extras: batchnorm
Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 6s - loss: 5.8844 - val_loss: 3.1967

# training | RMSE: 4.3637, MAE: 3.2062, R2: 0.7416, EVS: 0.7429

# test | RMSE: 4.3226, MAE: 3.1967, R2: 0.7475, EVS: 0.7492

11 seconds.

26 | Fri Jan 26 14:01:26 2018 | lowest loss so far: 4.1348 (run 20)

iterations: 1

Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 3s - loss: 8.5316 - val_loss: 5.3772

# training | RMSE: 6.7129, MAE: 5.4154, R2: 0.3884, EVS: 0.4499

# test | RMSE: 6.6800, MAE: 5.3772, R2: 0.3970, EVS: 0.4599

6 seconds.

37 | Fri Jan 26 14:03:48 2018 | lowest loss so far: 4.0915 (run 29)

iterations: 1.0
{'batch_size': 16,
 'init': 'glorot_uniform',
 'loss': 'mean_squared_error',
 'n_layers': 2,
 'optimizer': 'adam',
 'scaler': 'StandardScaler',
 'shuffle': True}
layer 1 | size:  22 | activation: tanh    | extras: None
layer 2 | size:   8 | activation: relu    | extras: dropout
- rate: 15.0%
Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 12s - loss: 30.4216 - val_loss: 16.9313

# training | RMSE: 4.1744, MAE: 3.0260, R2: 0.7635, EVS: 0.7636

# test | RMSE: 4.1148, MAE: 3.0087, R2: 0.7712, EVS: 0.7713

18 seconds.

38 | Fri Jan 26 14:04:07 2018 | lowest loss so far: 

Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 5s - loss: 181.3253 - val_loss: 138.0308

# training | RMSE: 11.7601, MAE: 9.5078, R2: -0.8769, EVS: 0.0000

# test | RMSE: 11.7486, MAE: 9.4802, R2: -0.8654, EVS: 0.0000

10 seconds.

49 | Fri Jan 26 14:06:46 2018 | lowest loss so far: 4.0485 (run 47)

iterations: 1.0
{'batch_size': 128,
 'init': 'glorot_normal',
 'loss': 'mean_squared_error',
 'n_layers': 2,
 'optimizer': 'adam',
 'scaler': 'MaxAbsScaler',
 'shuffle': True}
layer 1 | size:  20 | activation: sigmoid | extras: dropout
- rate: 32.2%
layer 2 | size:  34 | activation: sigmoid | extras: batchnorm
Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 4s - loss: 155.0387 - val_loss: 79.4377

# training | RMSE: 8.9050, MAE: 7.5687, R2: -0.0762, EVS: 0.0053

# test | RMSE: 8.9128, MAE: 7.5901, R2: -0.0735, EVS: 0.0053

7 seconds.

50 | Fri Jan 26 14:06:54 2018 | lowest 

Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 7s - loss: 3.9420 - val_loss: 3.0135

# training | RMSE: 4.2721, MAE: 3.0345, R2: 0.7523, EVS: 0.7542

# test | RMSE: 4.2081, MAE: 3.0135, R2: 0.7607, EVS: 0.7632

13 seconds.

61 | Fri Jan 26 14:09:44 2018 | lowest loss so far: 4.0485 (run 47)

iterations: 1.0
{'batch_size': 64,
 'init': 'glorot_normal',
 'loss': 'mean_absolute_error',
 'n_layers': 2,
 'optimizer': 'adamax',
 'scaler': 'MinMaxScaler',
 'shuffle': False}
layer 1 | size:   8 | activation: tanh    | extras: dropout
- rate: 38.4%
layer 2 | size:  30 | activation: tanh    | extras: batchnorm
Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 6s - loss: 8.1919 - val_loss: 5.7728

# training | RMSE: 6.8012, MAE: 5.7552, R2: 0.3722, EVS: 0.3859

# test | RMSE: 6.8071, MAE: 5.7728, R2: 0.3738, EVS: 0.3870

11 seconds.

62 | Fri Jan 26 14:09:54 2018 | lowest loss so f

...


# test | RMSE: 4.0108, MAE: 2.9096, R2: 0.7826, EVS: 0.7827

299 seconds.

*** 3.7777777777777777 configurations x 27.0 iterations each

166 | Fri Jan 26 17:01:55 2018 | lowest loss so far: 3.9146 (run 119)

iterations: 27.0
{'batch_size': 32,
 'init': 'uniform',
 'loss': 'mean_squared_error',
 'n_layers': 3,
 'optimizer': 'adagrad',
 'scaler': 'RobustScaler',
 'shuffle': False}
layer 1 | size:  10 | activation: tanh    | extras: None
layer 2 | size:  38 | activation: relu    | extras: batchnorm
layer 3 | size:  22 | activation: relu    | extras: dropout
- rate: 24.0%
Train on 68126 samples, validate on 12023 samples
Epoch 1/27
68126/68126 [==============================] - 24s - loss: 27.4511 - val_loss: 18.0294
Epoch 2/27
68126/68126 [==============================] - 17s - loss: 20.8112 - val_loss: 17.6724
Epoch 3/27
68126/68126 [==============================] - 17s - loss: 20.0637 - val_loss: 17.5001
Epoch 4/27
68126/68126 [==============================] - 17s - loss: 19.3788 - 

68126/68126 [==============================] - 5s - loss: 36.8433 - val_loss: 115.1620
Epoch 5/9
68126/68126 [==============================] - 5s - loss: 31.9630 - val_loss: 97.6778
Epoch 6/9
68126/68126 [==============================] - 5s - loss: 27.6227 - val_loss: 87.6198
Epoch 7/9
68126/68126 [==============================] - 5s - loss: 25.6310 - val_loss: 84.6396
Epoch 8/9
68126/68126 [==============================] - 5s - loss: 24.2902 - val_loss: 82.5323
Epoch 9/9
68126/68126 [==============================] - 5s - loss: 23.7206 - val_loss: 78.8495

# training | RMSE: 8.8857, MAE: 7.2396, R2: -0.0715, EVS: 0.3659

# test | RMSE: 8.8797, MAE: 7.2317, R2: -0.0656, EVS: 0.3690

64 seconds.

174 | Fri Jan 26 17:38:13 2018 | lowest loss so far: 3.9146 (run 119)

iterations: 9.0
{'batch_size': 64,
 'init': 'normal',
 'loss': 'mean_squared_error',
 'n_layers': 1,
 'optimizer': 'adamax',
 'scaler': 'MinMaxScaler',
 'shuffle': False}
layer 1 | size:  35 | activation: tanh    | extra

Train on 68126 samples, validate on 12023 samples
Epoch 1/9
68126/68126 [==============================] - 26s - loss: 62.0895 - val_loss: 20.3318
Epoch 2/9
68126/68126 [==============================] - 18s - loss: 20.9594 - val_loss: 17.5184
Epoch 3/9
68126/68126 [==============================] - 18s - loss: 19.0415 - val_loss: 16.3757
Epoch 4/9
68126/68126 [==============================] - 18s - loss: 18.3023 - val_loss: 16.0693
Epoch 5/9
68126/68126 [==============================] - 18s - loss: 17.9905 - val_loss: 15.9694
Epoch 6/9
68126/68126 [==============================] - 18s - loss: 17.7842 - val_loss: 15.9123
Epoch 7/9
68126/68126 [==============================] - 18s - loss: 17.6209 - val_loss: 15.8796
Epoch 8/9
68126/68126 [==============================] - 18s - loss: 17.5611 - val_loss: 15.8284
Epoch 9/9
68126/68126 [==============================] - 18s - loss: 17.4744 - val_loss: 15.7167

# training | RMSE: 4.0236, MAE: 2.8802, R2: 0.7803, EVS: 0.7803

# test | RM

68126/68126 [==============================] - 13s - loss: 4.4066 - val_loss: 3.0611
Epoch 2/27
68126/68126 [==============================] - 5s - loss: 2.9260 - val_loss: 2.8889
Epoch 3/27
68126/68126 [==============================] - 5s - loss: 2.8779 - val_loss: 2.8139
Epoch 4/27
68126/68126 [==============================] - 5s - loss: 2.8598 - val_loss: 2.8205
Epoch 5/27
68126/68126 [==============================] - 5s - loss: 2.8493 - val_loss: 2.8254
Epoch 6/27
68126/68126 [==============================] - 5s - loss: 2.8411 - val_loss: 2.8544
Epoch 7/27
68126/68126 [==============================] - 5s - loss: 2.8326 - val_loss: 2.8666
Epoch 8/27
68126/68126 [==============================] - 5s - loss: 2.8228 - val_loss: 2.9229
Epoch 9/27
68126/68126 [==============================] - 5s - loss: 2.8163 - val_loss: 2.9067

# training | RMSE: 4.2235, MAE: 2.9238, R2: 0.7579, EVS: 0.7621

# test | RMSE: 4.1637, MAE: 2.9067, R2: 0.7657, EVS: 0.7691

70 seconds.

188 | Fri Jan 2

68126/68126 [==============================] - 482s - loss: 111.9410 - val_loss: 76.1668
Epoch 2/27
68126/68126 [==============================] - 11s - loss: 62.6867 - val_loss: 67.0977
Epoch 3/27
68126/68126 [==============================] - 9s - loss: 56.0291 - val_loss: 58.1376
Epoch 4/27
68126/68126 [==============================] - 9s - loss: 54.8009 - val_loss: 50.3968
Epoch 5/27
68126/68126 [==============================] - 9s - loss: 54.0834 - val_loss: 48.1469
Epoch 6/27
68126/68126 [==============================] - 10s - loss: 54.0149 - val_loss: 45.4319
Epoch 7/27
68126/68126 [==============================] - 9s - loss: 53.8191 - val_loss: 45.4983
Epoch 8/27
68126/68126 [==============================] - 9s - loss: 53.6967 - val_loss: 43.5229
Epoch 9/27
68126/68126 [==============================] - 9s - loss: 54.0852 - val_loss: 43.8322
Epoch 10/27
68126/68126 [==============================] - 8s - loss: 53.9037 - val_loss: 43.2690
Epoch 11/27
68126/68126 [==========

In [6]:
# Only parametric ReLU activations, truncated space based on previous results...

max_layers = 4
max_layer_size = 20

space = {
    'scaler': hp.choice( 's', ( 'StandardScaler', 'RobustScaler' )),
    'n_layers': hp.quniform( 'ls', 1, max_layers, 1 ),
    'init': hp.choice( 'i', ( 'glorot_uniform', 'glorot_normal', 'he_uniform', 'he_normal' )),
    'batch_size': hp.choice( 'bs', ( 16, 32, 64 )),
    'shuffle': hp.choice( 'sh', ( False, True )),
    'loss': hp.choice( 'l', ( 'mean_absolute_error', 'mean_squared_error' )),
    'optimizer': hp.choice( 'o', ( 'rmsprop', 'adagrad', 'adam' ))        
}

# for each hidden layer, we choose size and extras individually
for i in range( 1, max_layers + 1 ):
    space[ 'layer_{}_size'.format( i )] = hp.quniform( 'ls{}'.format( i ), 
        2, max_layer_size, 1 )
    space[ 'layer_{}_extras'.format( i )] = hp.choice( 'e{}'.format( i ), ( 
        { 'name': 'dropout', 'rate': hp.uniform( 'd{}'.format( i ), 0.1, 0.5 )}, 
        { 'name': 'batchnorm' },
        { 'name': None } ))    
    
def get_params():
    params = sample( space )
    return handle_integers( params )

# print hidden layers config in readable way
def print_layers( params ):
    for i in range( 1, params['n_layers'] + 1 ):
        print("layer {} | size: {:>3} | extras: {}".format( i,
            params['layer_{}_size'.format( i )], 
            params['layer_{}_extras'.format( i )]['name'] ))
        if params['layer_{}_extras'.format( i )]['name'] == 'dropout':
            print("- rate: {:.1%}".format( params['layer_{}_extras'.format( i )]['rate'] ))

def print_params( params ):
    pprint({ k: v for k, v in params.items() if not k.startswith( 'layer_' )})
    print_layers( params )

def try_params( n_iterations, params ):
    
    print("iterations:", n_iterations)
    print_params( params )
    
    y_train = data['y_train']
    y_test = data['y_test']
    
    if params['scaler']:
        scaler = eval( "{}()".format( params['scaler'] ))
        x_train_ = scaler.fit_transform( data['x_train'].astype( float ))
        x_test_ = scaler.transform( data['x_test'].astype( float ))
    else:
        x_train_ = data['x_train']
        x_test_ = data['x_test']
        
    input_dim = x_train_.shape[1]

    model = Sequential()
    model.add( Dense( params['layer_1_size'], kernel_initializer = params['init'], 
        input_dim = input_dim ))
    model.add( PReLU( alpha_initializer=params['init'] ))
    
    for i in range( int( params['n_layers'] ) - 1 ):
        
        extras = 'layer_{}_extras'.format( i + 1 )
        
        if params[extras]['name'] == 'dropout':
            model.add( Dropout( params[extras]['rate'] ))
        elif params[extras]['name'] == 'batchnorm':
            model.add( BatchNorm())
            
        model.add( Dense( params['layer_{}_size'.format( i + 2 )], kernel_initializer = params['init'] ))
        model.add( PReLU( alpha_initializer=params['init'] ))
           
    model.add( Dense( 1, kernel_initializer = params['init'], activation = 'linear' ))

    model.compile( optimizer = params['optimizer'], loss = params['loss'] )
    
    validation_data = ( x_test_, y_test )

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 5, verbose = 0 )
    
    history = model.fit( x_train_, y_train,
        epochs = int( round( n_iterations )),
        batch_size = params['batch_size'], 
        shuffle = params['shuffle'], 
        validation_data = validation_data, 
        callbacks = [ early_stopping ])    
    
    p = model.predict( x_train_, batch_size = params['batch_size'] )

    mse = MSE( y_train, p )
    rmse = sqrt( mse )
    mae = MAE( y_train, p )
    r2 = R2( y_train, p )
    evs = EVS( y_train, p )

    print("\n# training | RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}".format( rmse, mae, r2, evs ))

    p = model.predict( x_test_, batch_size = params['batch_size'] )
    
    mse = MSE( y_test, p )
    rmse = sqrt( mse )
    mae = MAE( y_test, p )
    r2 = R2( y_test, p )
    evs = EVS( y_test, p )

    print("\n# test | RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}".format( rmse, mae, r2, evs ))
    
    return { 'loss': rmse, 'rmse': rmse, 'mae': mae, 'r2':r2, 'evs':evs, 'early_stop': model.stop_training }


In [7]:
from hyperband import Hyperband

output_file = 'results.pkl'
print("Will save results to", output_file)

hb = Hyperband( get_params, try_params )
results = hb.run( skip_last = 1 )

print("{} total, best:\n".format( len( results )))

for r in sorted( results, key = lambda x: x['loss'] )[:5]:
    print("loss: {:.2%} | {} seconds | {:.1f} iterations | run {} ".format( 
        r['loss'], r['seconds'], r['iterations'], r['counter'] ))
    pprint( r['params'] )

print("saving...")

with open( output_file, 'wb' ) as f:
    pickle.dump( results, f )

Will save results to results.pkl

*** 81 configurations x 1.0 iterations each

1 | Wed Jan 31 12:29:31 2018 | lowest loss so far: inf (run -1)

iterations: 1.0
{'batch_size': 16,
 'init': 'he_uniform',
 'loss': 'mean_absolute_error',
 'n_layers': 4,
 'optimizer': 'adagrad',
 'scaler': 'StandardScaler',
 'shuffle': True}
layer 1 | size:  12 | extras: dropout
- rate: 16.1%
layer 2 | size:  17 | extras: batchnorm
layer 3 | size:  10 | extras: batchnorm
layer 4 | size:   3 | extras: dropout
- rate: 30.9%
Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 16s - loss: 4.6662 - val_loss: 2.3869

# training | RMSE: 3.9801, MAE: 2.3827, R2: 0.7850, EVS: 0.7856

# test | RMSE: 4.2471, MAE: 2.3869, R2: 0.7566, EVS: 0.7570

25 seconds.

2 | Wed Jan 31 12:29:55 2018 | lowest loss so far: 4.2471 (run 1)

iterations: 1.0
{'batch_size': 64,
 'init': 'he_uniform',
 'loss': 'mean_squared_error',
 'n_layers': 2,
 'optimizer': 'adagrad',
 'scaler': '

68126/68126 [==============================] - 5s - loss: 37.1831 - val_loss: 22.4709

# training | RMSE: 4.6161, MAE: 3.6804, R2: 0.7107, EVS: 0.8026

# test | RMSE: 4.7403, MAE: 3.7309, R2: 0.6968, EVS: 0.7887

10 seconds.

14 | Wed Jan 31 12:32:19 2018 | lowest loss so far: 3.1851 (run 12)

iterations: 1.0
{'batch_size': 16,
 'init': 'glorot_normal',
 'loss': 'mean_absolute_error',
 'n_layers': 2,
 'optimizer': 'adam',
 'scaler': 'RobustScaler',
 'shuffle': True}
layer 1 | size:  14 | extras: dropout
- rate: 49.4%
layer 2 | size:  16 | extras: batchnorm
Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 11s - loss: 3.8200 - val_loss: 2.5470

# training | RMSE: 3.7504, MAE: 2.5244, R2: 0.8091, EVS: 0.8289

# test | RMSE: 3.9136, MAE: 2.5470, R2: 0.7933, EVS: 0.8130

18 seconds.

15 | Wed Jan 31 12:32:37 2018 | lowest loss so far: 3.1851 (run 12)

iterations: 1.0
{'batch_size': 64,
 'init': 'he_uniform',
 'loss': 'mean_absolute_e

68126/68126 [==============================] - 6s - loss: 6.1561 - val_loss: 1.8300

# training | RMSE: 3.4623, MAE: 1.8124, R2: 0.8373, EVS: 0.8376

# test | RMSE: 3.6972, MAE: 1.8300, R2: 0.8155, EVS: 0.8158

11 seconds.

40 | Wed Jan 31 12:37:44 2018 | lowest loss so far: 3.1851 (run 12)

iterations: 1.0
{'batch_size': 32,
 'init': 'he_uniform',
 'loss': 'mean_squared_error',
 'n_layers': 2,
 'optimizer': 'adam',
 'scaler': 'StandardScaler',
 'shuffle': False}
layer 1 | size:   2 | extras: batchnorm
layer 2 | size:  14 | extras: batchnorm
Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 10s - loss: 55.1779 - val_loss: 10.8139

# training | RMSE: 3.2185, MAE: 1.9353, R2: 0.8594, EVS: 0.8607

# test | RMSE: 3.2884, MAE: 1.9496, R2: 0.8541, EVS: 0.8554

17 seconds.

41 | Wed Jan 31 12:38:01 2018 | lowest loss so far: 3.1851 (run 12)

iterations: 1.0
{'batch_size': 32,
 'init': 'he_normal',
 'loss': 'mean_absolute_error',
 'n_lay

Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 9s - loss: 33.5354 - val_loss: 17.6682

# training | RMSE: 3.9927, MAE: 2.6590, R2: 0.7836, EVS: 0.7939

# test | RMSE: 4.2034, MAE: 2.6717, R2: 0.7616, EVS: 0.7722

16 seconds.

54 | Wed Jan 31 12:41:51 2018 | lowest loss so far: 3.1851 (run 12)

iterations: 1.0
{'batch_size': 32,
 'init': 'he_normal',
 'loss': 'mean_squared_error',
 'n_layers': 2,
 'optimizer': 'adagrad',
 'scaler': 'StandardScaler',
 'shuffle': True}
layer 1 | size:  17 | extras: None
layer 2 | size:  15 | extras: dropout
- rate: 18.9%
Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 8s - loss: 23.7235 - val_loss: 16.2404

# training | RMSE: 3.7953, MAE: 2.3664, R2: 0.8045, EVS: 0.8047

# test | RMSE: 4.0299, MAE: 2.3844, R2: 0.7809, EVS: 0.7812

15 seconds.

55 | Wed Jan 31 12:42:06 2018 | lowest loss so far: 3.1851 (run 12)

iterations: 1.0
{'batch_siz

Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 12s - loss: 45.8612 - val_loss: 12.4372

# training | RMSE: 3.3569, MAE: 1.9052, R2: 0.8470, EVS: 0.8481

# test | RMSE: 3.5266, MAE: 1.9294, R2: 0.8322, EVS: 0.8332

21 seconds.

67 | Wed Jan 31 12:46:20 2018 | lowest loss so far: 3.1851 (run 12)

iterations: 1.0
{'batch_size': 64,
 'init': 'he_uniform',
 'loss': 'mean_squared_error',
 'n_layers': 2,
 'optimizer': 'rmsprop',
 'scaler': 'RobustScaler',
 'shuffle': True}
layer 1 | size:  18 | extras: None
layer 2 | size:  15 | extras: batchnorm
Train on 68126 samples, validate on 12023 samples
Epoch 1/1
68126/68126 [==============================] - 6s - loss: 51.6631 - val_loss: 13.4378

# training | RMSE: 3.4178, MAE: 1.8455, R2: 0.8414, EVS: 0.8416

# test | RMSE: 3.6658, MAE: 1.8672, R2: 0.8187, EVS: 0.8189

12 seconds.

68 | Wed Jan 31 12:46:32 2018 | lowest loss so far: 3.1851 (run 12)

iterations: 1.0
{'batch_size': 64,
 'in

...

Train on 68126 samples, validate on 12023 samples
Epoch 1/9
68126/68126 [==============================] - 45s - loss: 5.1933 - val_loss: 1.9096
Epoch 2/9
68126/68126 [==============================] - 37s - loss: 2.6940 - val_loss: 1.8732
Epoch 3/9
68126/68126 [==============================] - 37s - loss: 2.6726 - val_loss: 1.8551
Epoch 4/9
68126/68126 [==============================] - 38s - loss: 2.6598 - val_loss: 1.8448
Epoch 5/9
68126/68126 [==============================] - 37s - loss: 2.6506 - val_loss: 1.8359
Epoch 6/9
68126/68126 [==============================] - 37s - loss: 2.6435 - val_loss: 1.8300
Epoch 7/9
68126/68126 [==============================] - 37s - loss: 2.6377 - val_loss: 1.8248
Epoch 8/9
68126/68126 [==============================] - 37s - loss: 2.6328 - val_loss: 1.8199
Epoch 9/9
68126/68126 [==============================] - 37s - loss: 2.6286 - val_loss: 1.8163

# training | RMSE: 3.4997, MAE: 1.7918, R2: 0.8337, EVS: 0.8342

# test | RMSE: 3.7498, MAE: 1

Train on 68126 samples, validate on 12023 samples
Epoch 1/9
68126/68126 [==============================] - 19s - loss: 65.3991 - val_loss: 11.2795
Epoch 2/9
68126/68126 [==============================] - 10s - loss: 12.2433 - val_loss: 10.7924
Epoch 3/9
68126/68126 [==============================] - 10s - loss: 11.2458 - val_loss: 10.8302
Epoch 4/9
68126/68126 [==============================] - 10s - loss: 10.8062 - val_loss: 10.8389
Epoch 5/9
68126/68126 [==============================] - 10s - loss: 10.6359 - val_loss: 10.8530
Epoch 6/9
68126/68126 [==============================] - 10s - loss: 10.5336 - val_loss: 10.6257
Epoch 7/9
68126/68126 [==============================] - 10s - loss: 10.4480 - val_loss: 10.6519
Epoch 8/9
68126/68126 [==============================] - 10s - loss: 10.3651 - val_loss: 10.5594
Epoch 9/9
68126/68126 [==============================] - 10s - loss: 10.2956 - val_loss: 10.5779

# training | RMSE: 3.1723, MAE: 1.9314, R2: 0.8634, EVS: 0.8659

# test | RM

Train on 68126 samples, validate on 12023 samples
Epoch 1/9
68126/68126 [==============================] - 45s - loss: 3.2614 - val_loss: 1.7678
Epoch 2/9
68126/68126 [==============================] - 35s - loss: 1.7294 - val_loss: 1.7375
Epoch 3/9
68126/68126 [==============================] - 36s - loss: 1.6995 - val_loss: 1.7041
Epoch 4/9
68126/68126 [==============================] - 35s - loss: 1.6732 - val_loss: 1.6861
Epoch 5/9
68126/68126 [==============================] - 35s - loss: 1.6625 - val_loss: 1.6816
Epoch 6/9
68126/68126 [==============================] - 35s - loss: 1.6589 - val_loss: 1.6773
Epoch 7/9
68126/68126 [==============================] - 35s - loss: 1.6572 - val_loss: 1.6816
Epoch 8/9
68126/68126 [==============================] - 35s - loss: 1.6553 - val_loss: 1.6820
Epoch 9/9
68126/68126 [==============================] - 35s - loss: 1.6557 - val_loss: 1.6897

# training | RMSE: 3.3815, MAE: 1.6654, R2: 0.8448, EVS: 0.8448

# test | RMSE: 3.6010, MAE: 1

68126/68126 [==============================] - 22s - loss: 2.1295 - val_loss: 1.6864
Epoch 12/27
68126/68126 [==============================] - 22s - loss: 2.1263 - val_loss: 1.6856
Epoch 13/27
68126/68126 [==============================] - 22s - loss: 2.0903 - val_loss: 1.6842
Epoch 14/27
68126/68126 [==============================] - 22s - loss: 2.1044 - val_loss: 1.6782
Epoch 15/27
68126/68126 [==============================] - 22s - loss: 2.0948 - val_loss: 1.6742
Epoch 16/27
68126/68126 [==============================] - 22s - loss: 2.0721 - val_loss: 1.6736
Epoch 17/27
68126/68126 [==============================] - 22s - loss: 2.0830 - val_loss: 1.6777
Epoch 18/27
68126/68126 [==============================] - 22s - loss: 2.0697 - val_loss: 1.6717
Epoch 19/27
68126/68126 [==============================] - 22s - loss: 2.0659 - val_loss: 1.6743
Epoch 20/27
68126/68126 [==============================] - 22s - loss: 2.0693 - val_loss: 1.6773
Epoch 21/27
68126/68126 [=================

68126/68126 [==============================] - 9s - loss: 11.2783 - val_loss: 12.7401
Epoch 13/27
68126/68126 [==============================] - 9s - loss: 11.0946 - val_loss: 12.5081
Epoch 14/27
68126/68126 [==============================] - 9s - loss: 10.9079 - val_loss: 12.4917
Epoch 15/27
68126/68126 [==============================] - 9s - loss: 10.7871 - val_loss: 12.2320
Epoch 16/27
68126/68126 [==============================] - 9s - loss: 10.6795 - val_loss: 11.9357
Epoch 17/27
68126/68126 [==============================] - 9s - loss: 10.5187 - val_loss: 11.7037
Epoch 18/27
68126/68126 [==============================] - 9s - loss: 10.4157 - val_loss: 11.5625
Epoch 19/27
68126/68126 [==============================] - 9s - loss: 10.3256 - val_loss: 11.3545
Epoch 20/27
68126/68126 [==============================] - 9s - loss: 10.2660 - val_loss: 11.3164
Epoch 21/27
68126/68126 [==============================] - 9s - loss: 10.1687 - val_loss: 11.1626
Epoch 22/27
68126/68126 [=======